# ProphetGP 사용 예시

이 노트북은 ProphetGP의 핵심 기능(학습, 후보 추천, 데이터셋 추가)을 빠르게 실행해보는 예시입니다.

In [ ]:
# 필요 시 주석 해제 후 설치
# !pip install -e .[dev]

In [2]:
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# 노트북 실행 위치가 notebooks/여도 안전하게 프로젝트 루트를 찾는다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "sample_emission.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "new_batch.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config loaded:", config.model_dump())

FileNotFoundError: [Errno 2] No such file or directory: 'configs\\sample_emission.yaml'

In [ ]:
# 1) 사용 가능한 featuriser 확인
available_featurisers = pipeline.featurizers.available()
print("Available featurisers count:", len(available_featurisers))
print(available_featurisers[:20])  # 앞쪽 일부만 출력

In [ ]:
# 2) 학습
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])

In [ ]:
# 3) 다음 실험 조건 후보 추천
n_candidates = 5
candidates = pipeline.suggest_next_experiments(artifacts, n_candidates=n_candidates)
print("Candidates shape:", candidates.shape)
candidates

In [ ]:
# 4) 신규 배치 데이터 append
# 파일이 준비되어 있지 않으면 이 셀은 건너뛰세요.
merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
print("Merged rows:", len(merged))
print("Saved to:", MERGED_OUT_PATH)

## 입력 데이터 포맷 가이드

- `reactants`: 반응물 리스트 (`|` 구분)
- `target`: 예측/최적화 대상 물성값
- 그 외 컬럼: 반응 조건(문자열/숫자 모두 가능, 자동 타입 추론 + config override 지원)